# Data Quality Testing

## Objective

Validate Bronze, Silver and Gold layer datasets using automated data quality checks.

### Validation Types

- Row Count Validation
- Null Checks
- Duplicate Checks
- Business Rule Validation
- Reconciliation Checks

### Output

- workspace.default.data_quality_report

In [0]:
from pyspark.sql.functions import (
    lit,
    current_timestamp,
    count,
    col
)


## Load Project Tables

In [0]:
silver_trips = spark.table(
    "workspace.default.silver_trips"
)

silver_events = spark.table(
    "workspace.default.silver_ride_events"
)

gold_trip_lakeflow = spark.table(
    "workspace.default.gold_trip_lakeflow"
)

lakeflow_silver = spark.table(
    "workspace.default.silver_trip_lakeflow"
)

customers = spark.table(
    "workspace.default.dim_customers_scd2"
)

## Row Count Validation

In [0]:
row_count_results = []

tables = {
    "silver_trips": silver_trips,
    "silver_ride_events": silver_events,
    "gold_trip_lakeflow": gold_trip_lakeflow
}

for table_name, df in tables.items():

    row_count = df.count()

    status = "PASS" if row_count > 0 else "FAIL"

    row_count_results.append(
        (
            f"{table_name}_row_count",
            status,
            f"Row Count = {row_count}"
        )
    )

row_count_df = spark.createDataFrame(
    row_count_results,
    [
        "test_name",
        "status",
        "details"
    ]
)

## Null Check Validation

In [0]:
null_boroughs = (
    gold_trip_lakeflow
    .filter(
        col("pickup_borough").isNull()
    )
    .count()
)

status = (
    "PASS"
    if null_boroughs == 0
    else "FAIL"
)

null_check_df = spark.createDataFrame(
    [
        (
            "pickup_borough_null_check",
            status,
            f"Null Count = {null_boroughs}"
        )
    ],
    [
        "test_name",
        "status",
        "details"
    ]
)


## Duplicate Check Validation

In [0]:
duplicate_count = (
    silver_events
    .groupBy(
        "ride_id",
        "event_type"
    )
    .count()
    .filter(
        col("count") > 1
    )
    .count()
)

status = (
    "PASS"
    if duplicate_count == 0
    else "FAIL"
)

duplicate_check_df = spark.createDataFrame(
    [
        (
            "duplicate_ride_event_check",
            status,
            f"Duplicate Groups = {duplicate_count}"
        )
    ],
    [
        "test_name",
        "status",
        "details"
    ]
)

## Business Rule Validation

In [0]:
negative_fare_count = (
    silver_trips
    .filter(
        col("fare_amount") < 0
    )
    .count()
)

fare_status = (
    "PASS"
    if negative_fare_count == 0
    else "FAIL"
)

negative_fare_df = spark.createDataFrame(
    [
        (
            "negative_fare_check",
            fare_status,
            f"Negative Fare Records = {negative_fare_count}"
        )
    ],
    [
        "test_name",
        "status",
        "details"
    ]
)

In [0]:
zero_distance_count = (
    silver_trips
    .filter(
        col("trip_distance") <= 0
    )
    .count()
)

zero_distance_df = spark.createDataFrame(
    [
        (
            "zero_distance_check",
            "WARN",
            f"Zero Distance Trips = {zero_distance_count}"
        )
    ],
    [
        "test_name",
        "status",
        "details"
    ]
)

## Reconciliation Check

In [0]:
lakeflow_silver_count = (
    lakeflow_silver.count()
)

gold_trip_count = (
    gold_trip_lakeflow
    .agg(
        {"trip_count": "sum"}
    )
    .collect()[0][0]
)

status = (
    "PASS"
    if lakeflow_silver_count == gold_trip_count
    else "FAIL"
)

reconciliation_df = spark.createDataFrame(
    [
        (
            "lakeflow_reconciliation",
            status,
            (
                f"Silver={lakeflow_silver_count}, "
                f"Gold={gold_trip_count}"
            )
        )
    ],
    [
        "test_name",
        "status",
        "details"
    ]
)

## Combine Test Results Using UNION

In [0]:
quality_report_df = (
    row_count_df
    .union(null_check_df)
    .union(duplicate_check_df)
)

## Add Additional Checks Using UNION BY NAME

In [0]:
additional_checks_df = (
    negative_fare_df
    .unionByName(
        zero_distance_df
    )
    .unionByName(
        reconciliation_df
    )
)

quality_report_df = (
    quality_report_df
    .unionByName(
        additional_checks_df
    )
)

## Add Execution Metadata


In [0]:
quality_report_df = (
    quality_report_df
    .withColumn(
        "execution_timestamp",
        current_timestamp()
    )
)

In [0]:
display(
    quality_report_df
)

## Create Data Quality Report

In [0]:
quality_report_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.data_quality_report"
    )

## Validation

In [0]:
%sql
SELECT *
FROM workspace.default.data_quality_report

In [0]:
%sql
SELECT
    status,
    COUNT(*) AS total_tests
FROM workspace.default.data_quality_report
GROUP BY status

## Summary

### Data Quality Checks

✅ Row Count Validation

✅ Null Checks

✅ Duplicate Checks

✅ Business Rule Validation

✅ Reconciliation Checks

### Output

- workspace.default.data_quality_report

### Key Findings

- No negative fare records detected
- No duplicate ride event groups detected
- No null pickup borough values detected
- Zero distance trips identified and reported as warnings
- Lakeflow reconciliation validated between Silver and Gold layers

✅ Testing Completed